In [142]:
from pathlib import Path
import polars as pl

import polars.selectors as cs

import re

ROOT = Path.cwd().parent if Path.cwd().name == 'script' else Path.cwd()

## 1.Data analysis and clean

use ClinVar variant_summary data

download: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz 
- select variant with protein vatiant info
- select review status above 2 stars
- dedup
- focus on missense mutation

In [143]:
Data_path = str(ROOT / 'variant_summary.txt.gz')

df = pl.read_csv(
    Data_path, 
    separator="\t",
    null_values=["", "na", "null","-"],
    infer_schema_length=100000,
    quote_char=None)


print(df.shape)

print(df.head())

(8981173, 43)
shape: (5, 43)
┌───────────┬────────────┬────────────┬────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ #AlleleID ┆ Type       ┆ Name       ┆ GeneID ┆ … ┆ ReviewSta ┆ SCVsForAg ┆ SCVsForAg ┆ SCVsForAg │
│ ---       ┆ ---        ┆ ---        ┆ ---    ┆   ┆ tusOncoge ┆ gregateGe ┆ gregateSo ┆ gregateOn │
│ i64       ┆ str        ┆ str        ┆ i64    ┆   ┆ nicity    ┆ rmlineCla ┆ maticClin ┆ cogenicit │
│           ┆            ┆            ┆        ┆   ┆ ---       ┆ ssi…      ┆ ica…      ┆ yCl…      │
│           ┆            ┆            ┆        ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│           ┆            ┆            ┆        ┆   ┆           ┆ str       ┆ str       ┆ str       │
╞═══════════╪════════════╪════════════╪════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 15041     ┆ Indel      ┆ NM_014855. ┆ 9907   ┆ … ┆ null      ┆ SCV001451 ┆ null      ┆ null      │
│           ┆            ┆ 3(AP5Z1):c ┆        ┆   ┆          

In [144]:
print(df.columns)

['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity', 'SCVsForAggregateGermlineClassification', 'SCVsForAggregateSomaticClinicalImpact', 'SCVsForAggregateOncogenicityClassification']


In [145]:
select_col = ['#AlleleID', 'Type', 'Name', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'PhenotypeList',  'OriginSimple', 'ReviewStatus']

df_select = df[select_col]

print(df_select.head(5))

print(df_select.shape)

shape: (5, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ #AlleleID ┆ Type      ┆ Name      ┆ GeneSymbo ┆ … ┆ ClinSigSi ┆ Phenotype ┆ OriginSim ┆ ReviewSt │
│ ---       ┆ ---       ┆ ---       ┆ l         ┆   ┆ mple      ┆ List      ┆ ple       ┆ atus     │
│ i64       ┆ str       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆ str       ┆   ┆ i64       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 15041     ┆ Indel     ┆ NM_014855 ┆ AP5Z1     ┆ … ┆ 1         ┆ Hereditar ┆ germline  ┆ criteria │
│           ┆           ┆ .3(AP5Z1) ┆           ┆   ┆           ┆ y spastic ┆           ┆ provided │
│           ┆           ┆ :c.80_83d ┆           ┆   ┆           ┆ paraplegi ┆           ┆ ,        │
│           ┆           ┆ eli…      ┆           ┆   ┆           ┆ a …       

### create protein variant
正则化提取蛋白质突变

In [146]:
# extract the protein variant from the Name column
df_select = df_select.with_columns(
    pl.col("Name")
    .str.extract(r"\(p\.([^)]+)\)")
    .alias("protein_variant")
)

print(df_select.head(5))

shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ #AlleleID ┆ Type      ┆ Name      ┆ GeneSymbo ┆ … ┆ Phenotype ┆ OriginSim ┆ ReviewSta ┆ protein_ │
│ ---       ┆ ---       ┆ ---       ┆ l         ┆   ┆ List      ┆ ple       ┆ tus       ┆ variant  │
│ i64       ┆ str       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 15041     ┆ Indel     ┆ NM_014855 ┆ AP5Z1     ┆ … ┆ Hereditar ┆ germline  ┆ criteria  ┆ Arg27_Il │
│           ┆           ┆ .3(AP5Z1) ┆           ┆   ┆ y spastic ┆           ┆ provided, ┆ e28delin │
│           ┆           ┆ :c.80_83d ┆           ┆   ┆ paraplegi ┆           ┆ multiple  ┆ sLeuLeuT │
│           ┆           ┆ eli…      ┆           ┆   ┆ a …       ┆           

In [147]:
# total genesymbol

print(df_select["GeneSymbol"].n_unique())


# protein variant count

print(df_select["protein_variant"].is_null().sum())

# select the rows with protein variant information

df_pro = df_select.filter(pl.col("protein_variant").is_not_null())

print(df_pro.shape)




41044
1890873
(7090300, 11)


### select the germline origin 
变异来自生殖遗传

In [148]:
# origin

print(df_pro["OriginSimple"].value_counts())

df_pro_germ = df_pro.filter(pl.col("OriginSimple") == "germline")

print(df_pro_germ.shape)

shape: (6, 2)
┌──────────────────┬─────────┐
│ OriginSimple     ┆ count   │
│ ---              ┆ ---     │
│ str              ┆ u32     │
╞══════════════════╪═════════╡
│ unknown          ┆ 76262   │
│ not applicable   ┆ 173523  │
│ germline/somatic ┆ 5662    │
│ not provided     ┆ 2375    │
│ germline         ┆ 6823197 │
│ somatic          ┆ 9281    │
└──────────────────┴─────────┘
(6823197, 11)


In [149]:
# review status 

print(df_pro_germ["ReviewStatus"].value_counts())


# 'ClinicalSignificance'

print(df_pro_germ["ClinicalSignificance"].value_counts())

shape: (8, 2)
┌─────────────────────────────────┬─────────┐
│ ReviewStatus                    ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ practice guideline              ┆ 80      │
│ criteria provided, conflicting… ┆ 283745  │
│ reviewed by expert panel        ┆ 35385   │
│ criteria provided, single subm… ┆ 5278733 │
│ no assertion criteria provided  ┆ 145624  │
│ criteria provided, multiple su… ┆ 1073875 │
│ no classification provided      ┆ 5539    │
│ no classifications from unflag… ┆ 216     │
└─────────────────────────────────┴─────────┘
shape: (82, 2)
┌─────────────────────────────────┬───────┐
│ ClinicalSignificance            ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Conflicting classifications of… ┆ 34    │
│ Conflicting classifications of… ┆ 6     │
│ Pathogenic/Likely

### ReviewStatus select

依据 ClinVar 官网说明，选取 2 star 及以上的条目

In [150]:
# select status above 2 stars

df_pro_review = df_pro_germ.filter(
    pl.col("ReviewStatus").is_in([
        'criteria provided, multiple submitters, no conflicts',
        'reviewed by expert panel',
        'practice guideline'
    ])
)

print(df_pro_review.shape)

(1109340, 11)


In [151]:
print(df_pro_review["Type"].value_counts())

shape: (8, 2)
┌───────────────────────────┬─────────┐
│ Type                      ┆ count   │
│ ---                       ┆ ---     │
│ str                       ┆ u32     │
╞═══════════════════════════╪═════════╡
│ single nucleotide variant ┆ 1029228 │
│ Deletion                  ┆ 42672   │
│ Microsatellite            ┆ 12170   │
│ Insertion                 ┆ 2392    │
│ Indel                     ┆ 4245    │
│ Duplication               ┆ 18167   │
│ Inversion                 ┆ 354     │
│ Variation                 ┆ 112     │
└───────────────────────────┴─────────┘


In [152]:
print(df_pro_review["ClinicalSignificance"].value_counts())

print(df_pro_review["ClinSigSimple"].value_counts())

shape: (50, 2)
┌─────────────────────────────────┬───────┐
│ ClinicalSignificance            ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Uncertain significance/VUS-hig… ┆ 24    │
│ Benign; association             ┆ 2     │
│ Likely pathogenic/Likely patho… ┆ 6     │
│ Conflicting classifications of… ┆ 2     │
│ Uncertain significance/VUS-low  ┆ 2     │
│ …                               ┆ …     │
│ Uncertain significance; drug r… ┆ 6     │
│ Pathogenic/Likely pathogenic/E… ┆ 2     │
│ Pathogenic/Likely pathogenic/P… ┆ 10    │
│ Benign/Likely benign; associat… ┆ 4     │
│ Benign/Likely benign; other; r… ┆ 2     │
└─────────────────────────────────┴───────┘
shape: (2, 2)
┌───────────────┬────────┐
│ ClinSigSimple ┆ count  │
│ ---           ┆ ---    │
│ i64           ┆ u32    │
╞═══════════════╪════════╡
│ 0             ┆ 946999 │
│ 1             ┆ 162341 │
└───────────────┴────────┘


In [153]:
print(df_pro_review.columns)

['#AlleleID', 'Type', 'Name', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'PhenotypeList', 'OriginSimple', 'ReviewStatus', 'protein_variant']


### 提取 transcript 信息用于 ref seq 查找

In [154]:
# extract the transcript variant from the Name column

df_pro_review = df_pro_review.with_columns(
    pl.col("Name")
    .str.extract(r"(NM_\d+\.\d+)").alias("refseq")
)


df_pro_review.head(5)


#AlleleID,Type,Name,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeList,OriginSimple,ReviewStatus,protein_variant,refseq
i64,str,str,str,str,str,i64,str,str,str,str,str
15041,"""Indel""","""NM_014855.3(AP5Z1):c.80_83deli…","""AP5Z1""","""HGNC:22197""","""Pathogenic/Likely pathogenic""",1,"""Hereditary spastic paraplegia …","""germline""","""criteria provided, multiple su…","""Arg27_Ile28delinsLeuLeuTer""","""NM_014855.3"""
15041,"""Indel""","""NM_014855.3(AP5Z1):c.80_83deli…","""AP5Z1""","""HGNC:22197""","""Pathogenic/Likely pathogenic""",1,"""Hereditary spastic paraplegia …","""germline""","""criteria provided, multiple su…","""Arg27_Ile28delinsLeuLeuTer""","""NM_014855.3"""
15044,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.694C>T …","""FOXRED1""","""HGNC:26927""","""Pathogenic""",1,"""Mitochondrial complex I defici…","""germline""","""criteria provided, multiple su…","""Gln232Ter""","""NM_017547.4"""
15044,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.694C>T …","""FOXRED1""","""HGNC:26927""","""Pathogenic""",1,"""Mitochondrial complex I defici…","""germline""","""criteria provided, multiple su…","""Gln232Ter""","""NM_017547.4"""
15054,"""single nucleotide variant""","""NM_000410.4(HFE):c.157G>A (p.V…","""HFE""","""HGNC:4886""","""Uncertain significance""",0,"""HFE POLYMORPHISM|Hemochromatos…","""germline""","""criteria provided, multiple su…","""Val53Met""","""NM_000410.4"""


In [155]:
print(df_pro_review["GeneSymbol"].value_counts())
print(df_pro_review["#AlleleID"].value_counts())

shape: (10_244, 2)
┌────────────┬───────┐
│ GeneSymbol ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ EML6       ┆ 12    │
│ OPN1MW2    ┆ 2     │
│ NOX1       ┆ 12    │
│ CTNNB1     ┆ 194   │
│ SLC7A14    ┆ 230   │
│ …          ┆ …     │
│ SAE1       ┆ 2     │
│ CBLN2      ┆ 4     │
│ KLC3       ┆ 2     │
│ MYO3A      ┆ 352   │
│ CD160      ┆ 2     │
└────────────┴───────┘
shape: (554_603, 2)
┌───────────┬───────┐
│ #AlleleID ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═══════════╪═══════╡
│ 2798199   ┆ 2     │
│ 2956806   ┆ 2     │
│ 835749    ┆ 2     │
│ 907633    ┆ 2     │
│ 642639    ┆ 2     │
│ …         ┆ …     │
│ 923082    ┆ 2     │
│ 473795    ┆ 2     │
│ 1150565   ┆ 2     │
│ 918317    ┆ 2     │
│ 907957    ┆ 2     │
└───────────┴───────┘


### dedup using Allele ID
由于每个变异有两种坐标表示，因此重复

In [156]:
# dedup using AlleleID, random select one

df_dedup = df_pro_review.unique(
    subset=["#AlleleID"],
    keep="first",
    maintain_order=True
)
print(df_dedup.shape)

print(df_dedup['Type'].value_counts())

(554603, 12)
shape: (8, 2)
┌───────────────────────────┬────────┐
│ Type                      ┆ count  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ Indel                     ┆ 2123   │
│ Deletion                  ┆ 21340  │
│ Inversion                 ┆ 177    │
│ Duplication               ┆ 9084   │
│ Variation                 ┆ 56     │
│ Insertion                 ┆ 1196   │
│ Microsatellite            ┆ 6082   │
│ single nucleotide variant ┆ 514545 │
└───────────────────────────┴────────┘


### 选择单氨基酸替换类型的突变

In [157]:
df_missense = df_dedup.filter(
    # 匹配  3 字母单氨基酸替换
    pl.col("protein_variant")
      .str.strip_chars()
      .str.contains(r"^[A-Z][a-z]{2}\d+[A-Z][a-z]{2}$")
    # 排除终止突变，如 "Gln232Ter"
    & ~pl.col("protein_variant")
         .str.strip_chars()
         .str.ends_with("Ter")
)

print(df_missense.shape)
print(df_missense["Type"].value_counts())
print(df_missense["protein_variant"].head(10))

print(df_missense["ClinSigSimple"].value_counts())


(341131, 12)
shape: (4, 2)
┌───────────────────────────┬────────┐
│ Type                      ┆ count  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ single nucleotide variant ┆ 340051 │
│ Deletion                  ┆ 1      │
│ Inversion                 ┆ 158    │
│ Indel                     ┆ 921    │
└───────────────────────────┴────────┘
shape: (10,)
Series: 'protein_variant' [str]
[
	"Val53Met"
	"Val59Met"
	"Gln283Pro"
	"Gly287Val"
	"Arg97Cys"
	"Arg70Pro"
	"Cys257Gly"
	"Glu526Lys"
	"Asp147His"
	"Gln265Pro"
]
shape: (2, 2)
┌───────────────┬────────┐
│ ClinSigSimple ┆ count  │
│ ---           ┆ ---    │
│ i64           ┆ u32    │
╞═══════════════╪════════╡
│ 1             ┆ 26218  │
│ 0             ┆ 314913 │
└───────────────┴────────┘


## 2.Use Comsic data for more negative sample

url: https://cancer.sanger.ac.uk/api/mono/products/v1/downloads/scripted?path=grch37/cmc/v104/CancerMutationCensus_AllData_Tsv_v104_GRCh37

- the origin from cosmic are all from somatic
- the MUTATION_SIGNIFICANCE_TIER shows that if the gene is a driver of cancer. We consider it as pathogenic.

In [158]:
Cosmic = pl.read_csv(
    str(ROOT / 'CancerMutationCensus_AllData_v104_GRCh37.tsv'),
    separator="\t",
    null_values=["", "na", "null","-"],
    infer_schema_length=100000,
    quote_char=None)

print(Cosmic.shape)

(5802907, 58)


In [159]:
Cosmic.head(5)

GENE_NAME,ACCESSION_NUMBER,ONC_TSG,CGC_TIER,MUTATION_URL,LEGACY_MUTATION_ID,Mutation CDS,Mutation AA,AA_MUT_START,AA_MUT_STOP,SHARED_AA,GENOMIC_WT_ALLELE_SEQ,GENOMIC_MUT_ALLELE_SEQ,AA_WT_ALLELE_SEQ,AA_MUT_ALLELE_SEQ,Mutation Description CDS,Mutation Description AA,ONTOLOGY_MUTATION_CODE,GENOMIC_MUTATION_ID,Mutation genome position GRCh37,Mutation genome position GRCh38,COSMIC_SAMPLE_TESTED,COSMIC_SAMPLE_MUTATED,DISEASE,WGS_DISEASE,EXAC_AF,EXAC_AFR_AF,EXAC_AMR_AF,EXAC_ADJ_AF,EXAC_EAS_AF,EXAC_FIN_AF,EXAC_NFE_AF,EXAC_SAS_AF,GNOMAD_EXOMES_AF,GNOMAD_EXOMES_AFR_AF,GNOMAD_EXOMES_AMR_AF,GNOMAD_EXOMES_ASJ_AF,GNOMAD_EXOMES_EAS_AF,GNOMAD_EXOMES_FIN_AF,GNOMAD_EXOMES_NFE_AF,GNOMAD_EXOMES_SAS_AF,GNOMAD_GENOMES_AF,GNOMAD_GENOMES_AFR_AF,GNOMAD_GENOMES_AMI_AF,GNOMAD_GENOMES_AMR_AF,GNOMAD_GENOMES_ASJ_AF,GNOMAD_GENOMES_EAS_AF,GNOMAD_GENOMES_FIN_AF,GNOMAD_GENOMES_MID_AF,GNOMAD_GENOMES_NFE_AF,GNOMAD_GENOMES_SAS_AF,CLINVAR_CLNSIG,CLINVAR_TRAIT,GERP++_RS,MIN_SIFT_SCORE,MIN_SIFT_PRED,DNDS_DISEASE_QVAL_SIG,MUTATION_SIGNIFICANCE_TIER
str,str,str,i64,str,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,str,str
"""CDKL5""","""ENST00000379989.3""",null,null,"""https://cancer.sanger.ac.uk/co…","""COSM138150""","""c.1915G>A""","""p.A639T""",639,639,3,"""G""","""A""","""A""","""T""","""Substitution""","""Substitution - Missense""","""SO:0001583""","""COSV66112854""","""X:18622959-18622959""","""X:18604839-18604839""",56640,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5.83,0.0,"""D""",null,"""Other"""
"""CDKL5""","""ENST00000379989.3""",null,null,"""https://cancer.sanger.ac.uk/co…","""COSM182259""","""c.2624G>A""","""p.R875Q""",875,875,2,"""G""","""A""","""R""","""Q""","""Substitution""","""Substitution - Missense""","""SO:0001583""","""COSV66110116""","""X:18646618-18646618""","""X:18628498-18628498""",56640,2,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,4.81,0.0,"""D""",null,"""Other"""
"""CDKL5""","""ENST00000379989.3""",null,null,"""https://cancer.sanger.ac.uk/co…","""COSM224112""","""c.412C>T""","""p.P138S""",138,138,1,"""C""","""T""","""P""","""S""","""Substitution""","""Substitution - Missense""","""SO:0001583""","""COSV66113302""","""X:18600019-18600019""","""X:18581899-18581899""",56640,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6.07,0.001,"""D""",null,"""Other"""
"""CDKL5""","""ENST00000379989.3""",null,null,"""https://cancer.sanger.ac.uk/co…","""COSM388310""","""c.2081G>T""","""p.G694V""",694,694,1,"""G""","""T""","""G""","""V""","""Substitution""","""Substitution - Missense""","""SO:0001583""","""COSV66110478""","""X:18627619-18627619""","""X:18609499-18609499""",56640,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6.14,0.0,"""D""",null,"""Other"""
"""CDKL5""","""ENST00000379989.3""",null,null,"""https://cancer.sanger.ac.uk/co…","""COSM488226""","""c.1768G>A""","""p.E590K""",590,590,1,"""G""","""A""","""E""","""K""","""Substitution""","""Substitution - Missense""","""SO:0001583""","""COSV66110166""","""X:18622812-18622812""","""X:18604692-18604692""",56640,4,null,null,0.0000494,0.0,0.0004,0.0000684,0.0,0.0,0.0000208,0.0,0.0000382,0.0,0.0001,0.0,0.0,0.0,0.0000245,0.0,0.0000356,0.0,0.0,0.000094,0.0,0.0,0.0,0.0,0.0000563,0.0,"""Uncertain significance&Likely …","""Angelman syndrome-like&not pro…",6.03,0.001,"""D""",null,"""Other"""


In [160]:
select_col_cosmic =['GENE_NAME','ACCESSION_NUMBER','LEGACY_MUTATION_ID','Mutation AA','Mutation Description AA','GENOMIC_MUTATION_ID','MUTATION_SIGNIFICANCE_TIER']

cosmic_select = Cosmic[select_col_cosmic]

cosmic_select.head(5)

GENE_NAME,ACCESSION_NUMBER,LEGACY_MUTATION_ID,Mutation AA,Mutation Description AA,GENOMIC_MUTATION_ID,MUTATION_SIGNIFICANCE_TIER
str,str,str,str,str,str,str
"""CDKL5""","""ENST00000379989.3""","""COSM138150""","""p.A639T""","""Substitution - Missense""","""COSV66112854""","""Other"""
"""CDKL5""","""ENST00000379989.3""","""COSM182259""","""p.R875Q""","""Substitution - Missense""","""COSV66110116""","""Other"""
"""CDKL5""","""ENST00000379989.3""","""COSM224112""","""p.P138S""","""Substitution - Missense""","""COSV66113302""","""Other"""
"""CDKL5""","""ENST00000379989.3""","""COSM388310""","""p.G694V""","""Substitution - Missense""","""COSV66110478""","""Other"""
"""CDKL5""","""ENST00000379989.3""","""COSM488226""","""p.E590K""","""Substitution - Missense""","""COSV66110166""","""Other"""


In [161]:
print(cosmic_select["GENE_NAME"].value_counts())

print(cosmic_select["Mutation AA"].is_null().sum())

print(cosmic_select["MUTATION_SIGNIFICANCE_TIER"].value_counts())

shape: (19_996, 2)
┌───────────────┬───────┐
│ GENE_NAME     ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ G2E3          ┆ 279   │
│ LSM4          ┆ 65    │
│ ARNTL         ┆ 256   │
│ POLR2D        ┆ 60    │
│ DNAH10        ┆ 2465  │
│ …             ┆ …     │
│ LEP           ┆ 100   │
│ DNAJC25-GNG10 ┆ 43    │
│ C2orf91       ┆ 36    │
│ ITCH          ┆ 385   │
│ MIEN1         ┆ 53    │
└───────────────┴───────┘
0
shape: (4, 2)
┌────────────────────────────┬─────────┐
│ MUTATION_SIGNIFICANCE_TIER ┆ count   │
│ ---                        ┆ ---     │
│ str                        ┆ u32     │
╞════════════════════════════╪═════════╡
│ 2                          ┆ 401     │
│ 3                          ┆ 57319   │
│ Other                      ┆ 5743017 │
│ 1                          ┆ 2170    │
└────────────────────────────┴─────────┘


In [162]:
# exclude tier "Other"

cosmic_Tier = cosmic_select.filter(
    pl.col("MUTATION_SIGNIFICANCE_TIER") != "Other")

print(cosmic_Tier.shape)



(59890, 7)


In [163]:
print(cosmic_Tier["Mutation Description AA"].value_counts())

shape: (8, 2)
┌─────────────────────────────┬───────┐
│ Mutation Description AA     ┆ count │
│ ---                         ┆ ---   │
│ str                         ┆ u32   │
╞═════════════════════════════╪═══════╡
│ Insertion - Frameshift      ┆ 12519 │
│ Substitution - Nonsense     ┆ 21109 │
│ Nonstop extension           ┆ 6     │
│ Frameshift                  ┆ 51    │
│ Deletion - Frameshift       ┆ 20249 │
│ Complex - frameshift        ┆ 933   │
│ Substitution - Missense     ┆ 5019  │
│ Complex - insertion inframe ┆ 4     │
└─────────────────────────────┴───────┘


In [164]:
# only keep missense mutation in cosmic

cosmic_missense = cosmic_Tier.filter(
    pl.col("Mutation Description AA") == "Substitution - Missense")

print(cosmic_missense.shape)

(5019, 7)


## 3.HumpVar


Description: Index of human variants curated from literature reports
Name:        humsavar.txt
Release:     2026_02 of 10-Jun-2026



In [165]:
import pandas as pd
import polars as pl

# humsavar.txt 是固定宽度格式
colspecs = [
    (0, 9),    # gene name
    (12, 22),  # UniProt AC
    (23, 34),  # FTId
    (35, 49),  # AA change (p.His52Arg)
    (50, 58),  # Variant category (LP/P, LB/B, US)
    (59, 73),  # dbSNP
    (74, 95),  # Disease name
]

df_humsavar = pd.read_fwf(
    str(ROOT / 'humsavar.txt'),
    colspecs=colspecs,
    skiprows=44,      # 跳过头部说明
    header=None,
    names=["gene", "uniprot_id", "ft_id", "protein_change", "category", "dbsnp", "disease"]
)

# 转成 Polars
df_humsavar = pl.from_pandas(df_humsavar)

# 清洗：去掉无基因名 / 版权行，只保留三类标签
df_humsavar = df_humsavar.filter(
    (pl.col("gene") != "-") &
    pl.col("category").is_in(["LP/P", "LB/B", "US"])
)

# 二值标签：LP/P=1（致病），LB/B=0（良性），US=None（剔除）
df_humsavar = df_humsavar.with_columns(
    pl.when(pl.col("category") == "LP/P").then(1)
      .when(pl.col("category") == "LB/B").then(0)
      .otherwise(None)
      .cast(pl.Int64)
      .alias("label")
)

# 只保留有标签的错义突变
df_humsavar = df_humsavar.filter(
    pl.col("label").is_not_null() &
    pl.col("protein_change").str.contains(r"^p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}$")
)

print(df_humsavar.shape)
print(df_humsavar["category"].value_counts())
print(df_humsavar.head(10))

(73180, 8)
shape: (2, 2)
┌──────────┬───────┐
│ category ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ LB/B     ┆ 39947 │
│ LP/P     ┆ 33233 │
└──────────┴───────┘
shape: (10, 8)
┌───────┬────────────┬────────────┬────────────────┬──────────┬────────────┬─────────┬───────┐
│ gene  ┆ uniprot_id ┆ ft_id      ┆ protein_change ┆ category ┆ dbsnp      ┆ disease ┆ label │
│ ---   ┆ ---        ┆ ---        ┆ ---            ┆ ---      ┆ ---        ┆ ---     ┆ ---   │
│ str   ┆ str        ┆ str        ┆ str            ┆ str      ┆ str        ┆ str     ┆ i64   │
╞═══════╪════════════╪════════════╪════════════════╪══════════╪════════════╪═════════╪═══════╡
│ A1BG  ┆ P04217     ┆ VAR_018369 ┆ p.His52Arg     ┆ LB/B     ┆ rs893184   ┆ -       ┆ 0     │
│ A1BG  ┆ P04217     ┆ VAR_018370 ┆ p.His395Arg    ┆ LB/B     ┆ rs2241788  ┆ -       ┆ 0     │
│ A1CF  ┆ Q9NQ94     ┆ VAR_052201 ┆ p.Val555Met    ┆ LB/B     ┆ rs9073     ┆ -       ┆ 0     │
│ A1CF  ┆ Q9NQ94     ┆ VAR_05982

In [166]:
df_humsavar.head(10)

gene,uniprot_id,ft_id,protein_change,category,dbsnp,disease,label
str,str,str,str,str,str,str,i64
"""A1BG""","""P04217""","""VAR_018369""","""p.His52Arg""","""LB/B""","""rs893184""","""-""",0
"""A1BG""","""P04217""","""VAR_018370""","""p.His395Arg""","""LB/B""","""rs2241788""","""-""",0
"""A1CF""","""Q9NQ94""","""VAR_052201""","""p.Val555Met""","""LB/B""","""rs9073""","""-""",0
"""A1CF""","""Q9NQ94""","""VAR_059821""","""p.Ala558Ser""","""LB/B""","""rs11817448""","""-""",0
"""A2M""","""P01023""","""VAR_000012""","""p.Arg704His""","""LB/B""","""rs1800434""","""-""",0
"""A2M""","""P01023""","""VAR_000013""","""p.Cys972Tyr""","""LB/B""","""rs1800433""","""-""",0
"""A2M""","""P01023""","""VAR_000014""","""p.Ile1000Val""","""LB/B""","""rs669""","""-""",0
"""A2M""","""P01023""","""VAR_026820""","""p.Asn639Asp""","""LB/B""","""rs226405""","""-""",0
"""A2M""","""P01023""","""VAR_026821""","""p.Leu815Gln""","""LB/B""","""rs3180392""","""-""",0


## 4.Combine 3 dataset

In [167]:
aa3_to_aa1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C",
    "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H", "Ile": "I",
    "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P",
    "Ser": "S", "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
}

def protein_variant_to_one_letter(x):
    if x is None:
        return None

    x = str(x).strip()
    x = x.removeprefix("p.")

    # 已经是单字母格式，如 A639T
    if re.fullmatch(r"[A-Z]\d+[A-Z]", x):
        return x

    # 三字母格式，如 Arg875Gln
    m = re.fullmatch(r"([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", x)
    if m:
        ref, pos, alt = m.groups()
        return f"{aa3_to_aa1.get(ref, ref)}{pos}{aa3_to_aa1.get(alt, alt)}"

    return None

In [168]:
clinvar_final = df_missense.with_columns([
    pl.col("Name")
      .str.extract(r"^(NM_\d+\.\d+)")
      .alias("mapping_id"),

    pl.col("protein_variant")
      .map_elements(protein_variant_to_one_letter, return_dtype=pl.Utf8)
      .alias("protein_variant"),

    pl.lit("ClinVar").alias("source"),
]).select([
    pl.col("GeneSymbol").alias("gene_symbol"),
    pl.col("ClinSigSimple").cast(pl.Int64).alias("ClinicalSig"),
    pl.col("protein_variant"),
    pl.col("source"),
    pl.col("#AlleleID").cast(pl.Utf8).alias("database_id"),
    pl.col("mapping_id"),
]).filter(
    pl.col("protein_variant").is_not_null() &
    pl.col("ClinicalSig").is_in([0, 1]) &
    pl.col("mapping_id").is_not_null()
)

print(clinvar_final.shape)
clinvar_final.head()

(341130, 6)


gene_symbol,ClinicalSig,protein_variant,source,database_id,mapping_id
str,i64,str,str,str,str
"""HFE""",0,"""V53M""","""ClinVar""","""15054""","""NM_000410.4"""
"""HFE""",0,"""V59M""","""ClinVar""","""15055""","""NM_000410.4"""
"""HFE""",1,"""Q283P""","""ClinVar""","""15058""","""NM_000410.4"""
"""HOGA1""",1,"""G287V""","""ClinVar""","""15069""","""NM_138413.4"""
"""HOGA1""",1,"""R97C""","""ClinVar""","""15070""","""NM_138413.4"""


In [169]:
cosmic_final = cosmic_missense.with_columns([
    pl.col("Mutation AA")
      .map_elements(protein_variant_to_one_letter, return_dtype=pl.Utf8)
      .alias("protein_variant"),

    pl.lit(1).cast(pl.Int64).alias("ClinicalSig"),
    pl.lit("COSMIC").alias("source"),
]).select([
    pl.col("GENE_NAME").alias("gene_symbol"),
    pl.col("ClinicalSig"),
    pl.col("protein_variant"),
    pl.col("source"),
    pl.col("LEGACY_MUTATION_ID").cast(pl.Utf8).alias("database_id"),
    pl.col("ACCESSION_NUMBER").cast(pl.Utf8).alias("mapping_id"),
]).filter(
    pl.col("protein_variant").is_not_null() &
    pl.col("database_id").is_not_null() &
    pl.col("mapping_id").is_not_null()
)

print(cosmic_final.shape)
cosmic_final.head()

(4967, 6)


gene_symbol,ClinicalSig,protein_variant,source,database_id,mapping_id
str,i64,str,str,str,str
"""CDKL5""",1,"""S603F""","""COSMIC""","""COSM1559879""","""ENST00000379989.3"""
"""MYLK""",1,"""A1099T""","""COSMIC""","""COSM1037351""","""ENST00000360304.3"""
"""ARID1A""",1,"""R2236P""","""COSMIC""","""COSM907760""","""ENST00000324856.7"""
"""ATR""",1,"""S756F""","""COSMIC""","""COSM1559620""","""ENST00000350721.4"""
"""EPHA8""",1,"""Y855N""","""COSMIC""","""COSM1559284""","""ENST00000166244.3"""


In [170]:
humsavar_final = (
    df_humsavar
    .with_columns([
        pl.col("protein_change")
          .str.replace(r"^p\.", "")
          .map_elements(protein_variant_to_one_letter, return_dtype=pl.String)
          .alias("protein_variant"),
        pl.lit("humsavar").alias("source"),
    ])
    .select([
        pl.col("gene").alias("gene_symbol"),
        pl.col("label").cast(pl.Int64).alias("ClinicalSig"),
        pl.col("protein_variant"),
        pl.col("source"),
        pl.col("ft_id").cast(pl.String).alias("database_id"),
        pl.col("uniprot_id").cast(pl.String).alias("mapping_id"),
    ])
    .filter(
        pl.col("protein_variant").is_not_null()
        & pl.col("ClinicalSig").is_in([0, 1])
        & pl.col("database_id").is_not_null()
        & pl.col("mapping_id").is_not_null()
    )
)

print(humsavar_final.shape)
humsavar_final.head()

(73180, 6)


gene_symbol,ClinicalSig,protein_variant,source,database_id,mapping_id
str,i64,str,str,str,str
"""A1BG""",0,"""H52R""","""humsavar""","""VAR_018369""","""P04217"""
"""A1BG""",0,"""H395R""","""humsavar""","""VAR_018370""","""P04217"""
"""A1CF""",0,"""V555M""","""humsavar""","""VAR_052201""","""Q9NQ94"""
"""A1CF""",0,"""A558S""","""humsavar""","""VAR_059821""","""Q9NQ94"""
"""A2M""",0,"""R704H""","""humsavar""","""VAR_000012""","""P01023"""


In [171]:
merged_dataset = pl.concat(
    [clinvar_final, cosmic_final, humsavar_final],
    how="vertical"
)

merged_dataset = merged_dataset.unique(
    subset=["gene_symbol", "protein_variant", "source", "database_id"],
    keep="first",
    maintain_order=True
)

print(merged_dataset.shape)
print(merged_dataset["source"].value_counts())
print(merged_dataset["ClinicalSig"].value_counts())


merged_dataset.head(10)

# save the merged dataset to csv file
merged_dataset.write_csv("merged_dataset.csv")

(418251, 6)
shape: (3, 2)
┌──────────┬────────┐
│ source   ┆ count  │
│ ---      ┆ ---    │
│ str      ┆ u32    │
╞══════════╪════════╡
│ humsavar ┆ 72155  │
│ COSMIC   ┆ 4966   │
│ ClinVar  ┆ 341130 │
└──────────┴────────┘
shape: (2, 2)
┌─────────────┬────────┐
│ ClinicalSig ┆ count  │
│ ---         ┆ ---    │
│ i64         ┆ u32    │
╞═════════════╪════════╡
│ 0           ┆ 354838 │
│ 1           ┆ 63413  │
└─────────────┴────────┘
